# 00 — Build the AML images

**Run this before `e2e_austria_aml.ipynb`.** Since spec 56 the image recipe is `fsd.image`,
not this notebook: what you see below is two declarations and two calls, because the
Dockerfile-writing, wheel-building and staleness-tracking logic that used to live here now
lives in `fsd.image`/`fsd.aml` where any consumer can import it too.

## Which image do you actually need?

fsd runs two node images, rebuilt on different schedules:

| image | used by | rebuild when |
|---|---|---|
| **`fsd-aml-env`** (Part A) | download shards, datacube builds, `create_training_data`'s flatten | the **fsd source** changed |
| **`fsd-infer-sklearn`** (Part B) | the `run_inference` fan-out, `verify_image`'s smoke job | the fsd source changed, **or** your model's runtime deps changed |

**Nothing to decide about staleness anymore.** `fsd.aml.ensure_environment` is
check-then-build (spec 56 D4): it digests the definition you pass, looks it up in the image
registry, confirms the AML asset it names still exists, and only builds if either check
misses. Running a cell that already matches what's registered costs one registry lookup and
one `az ml environment show` — safe to re-run every time, unlike the old notebook's
`az ml environment create`, which minted a new version unconditionally.

### You do NOT need to rebuild anything when

- you **retrained your model** — it rides in the bundle
- you **edited your adapter code** — it rides in the bundle too (spec 44)

Only a change to fsd itself, or to your dependency *family* (sklearn → torch), needs an image.

**First time here?** Run every cell top to bottom. **Coming back?** Same — `ensure_environment`
tells you whether it built or reused.

## 0.1 — Prerequisites

**Azure CLI**, logged in, with the `ml` extension:

```bash
az login                       # run in a terminal, not here — it opens a browser
az extension add -n ml         # once per machine
```

**Your Azure coordinates in `~/.config/fsd/config.toml`.** This notebook is committed to a
public repo, so it holds no resource group, workspace, subscription id, cluster name or
storage URL. It reads them through `fsd.config.load()`:

```bash
fsd init                       # interactive; fill in all values once, including
                                # image_registry (spec 55 D2)
```

Every variable is documented in [`../docs/reference/environment.md`](../docs/reference/environment.md).
Concrete values come from your platform admin — this repo never carries them.

`az login` is interactive, so run it in a terminal. The cell below only *checks* the result.

In [ ]:
# Which subscription am I about to build into? It must be the one holding the
# workspace below -- `az ml environment create` has no subscription flag, it uses this one.
#
# Prints a yes/no rather than the id: this notebook is committed, and a saved output
# carrying a subscription id or a user principal is exactly the leak that keeps notebooks
# out of public repos. Run `az account show` in a terminal when you need the detail.
import subprocess as _sp

_acct = _sp.run(["az", "account", "show", "--query", "id", "-o", "tsv"],
                capture_output=True, text=True)
print("az logged in:", bool(_acct.stdout.strip()))
if not _acct.stdout.strip():
    print("  ->", _acct.stderr.strip()[:300] or "run `az login` in a terminal")


## 0.2 — Configure

The only cell in this notebook you edit.

In [ ]:
import pathlib

import fsd
import fsd.aml
from fsd.image import ImageDefinition

# This notebook lives in the fsd checkout -- a developer artifact, not something a
# consumer runs -- so its own location resolves the repo root directly (spec 54 D6).
REPO = pathlib.Path.cwd().parent
assert (REPO / "pyproject.toml").exists(), f"expected an fsd checkout at {REPO}"

# `fsd init` writes these to ~/.config/fsd/config.toml, outside this repo (spec 54).
# `image_registry` is spec 55 D2's optional key -- the concrete path lives in YOUR
# config, never in this file.
cfg = fsd.config.load()
AZ_RG, AZ_ML_WORKSPACE = cfg.resource_group, cfg.workspace
IMAGE_REGISTRY = cfg.image_registry
assert IMAGE_REGISTRY, "set image_registry in ~/.config/fsd/config.toml (fsd init) first"

# The developer's own checkout, as fsd built it (spec 56 D1/D5): `fsd="path:..."` is the
# one case ensure_environment builds a wheel for, hashing its CONTENT so an uncommitted
# edit changes the digest -- there is no separate git-dirty check to run first.
BASE = ImageDefinition(
    name="fsd-aml-env",
    fsd=f"path:{REPO}",
    # azure -> adlfs + azure-identity + azure-keyvault-secrets: blob I/O through the
    #          storage seam, managed-identity auth, Key Vault creds on the node.
    # mpc   -> planetary-computer: signs asset hrefs on the node, right before transfer.
    # NOT aml  -> azure-ai-ml is the DRIVER-side dispatch SDK; the node never submits jobs.
    # NOT grid -> s2/s2cell tile the ROI on the driver, before any job exists.
    extras=("azure", "mpc"),
)

print("resource group :", AZ_RG)
print("workspace      :", AZ_ML_WORKSPACE)
print("image registry :", IMAGE_REGISTRY)
print("repo           :", REPO)


---

# Part A — `fsd-aml-env` (general purpose)

Download shards, datacube builds, `create_training_data`'s flatten. **Rebuilds only when the
fsd source changed** -- `ensure_environment` checks that for you.

In [ ]:
result_a = fsd.aml.ensure_environment(
    BASE, registry=IMAGE_REGISTRY, resource_group=AZ_RG, workspace=AZ_ML_WORKSPACE,
    storage="azure",   # the registry is on blob: authenticate, never read it anonymously
)
# .ref is AML's version (what `environment=` wants); .registry_ref is the image
# registry's own numbering of DEFINITIONS (what verify_image(image_ref=) wants).
print(f"spec {result_a.digest[:13]}  ->  {result_a.ref}  "
      f"({'reusing' if result_a.reused else 'just built'}; "
      f"registry {result_a.registry_ref})")
if result_a.build_url:
    from IPython.display import Markdown, display
    display(Markdown(
        f"**Wait for `Build status: Succeeded` before using it** if this was just built -- "
        f"~10-20 min of ACR time, occasionally flaky.\n\n- [`{result_a.ref}`]({result_a.build_url})"
    ))


---

# Part B — `fsd-infer-sklearn` (inference)

The `run_inference` fan-out and `verify_image`'s smoke job. Generic per **dependency family**,
never per model (spec 44): the adapter's source rides inside the bundle, so this image copies
no adapter and sets no `PYTHONPATH`. Every sklearn/joblib model you ever bundle runs on this
one image.

**Rebuilds when** the fsd source changed, **or** when your model's runtime deps changed. For a
different family (torch, xgboost), `derive()` a new definition with different `extra_pip` and
a different `name`.

In [ ]:
INFER = BASE.derive(name="fsd-infer-sklearn", extra_pip=("scikit-learn", "joblib"))

result_b = fsd.aml.ensure_environment(
    INFER, registry=IMAGE_REGISTRY, resource_group=AZ_RG, workspace=AZ_ML_WORKSPACE,
    storage="azure",   # the registry is on blob: authenticate, never read it anonymously
)
# .ref is AML's version (what `environment=` wants); .registry_ref is the image
# registry's own numbering of DEFINITIONS (what verify_image(image_ref=) wants).
print(f"spec {result_b.digest[:13]}  ->  {result_b.ref}  "
      f"({'reusing' if result_b.reused else 'just built'}; "
      f"registry {result_b.registry_ref})")
if result_b.build_url:
    from IPython.display import Markdown, display
    display(Markdown(
        f"**Wait for `Build status: Succeeded` before using it** if this was just built -- "
        f"~10-20 min of ACR time, occasionally flaky.\n\n- [`{result_b.ref}`]({result_b.build_url})"
    ))


---

# Part C — Using these in the e2e notebook

**Nothing to paste.** `e2e_austria_aml.ipynb` calls `fsd.aml.ensure_environment` itself,
against the same `IMAGE_REGISTRY`, and gets the same answer this notebook just did (D7's
usability win: two notebooks asking the registry the same question, instead of one pasting
values into the other).

Both image builds must read `Build status: Succeeded` in Studio before you run the e2e
notebook.

## Troubleshooting

**`ensure_environment` built when I expected it to reuse.** Either the fsd source really did
change (for `fsd="path:..."`, that includes an uncommitted edit -- the digest is of wheel
*content*, not git state, spec 56 D5), or the registry's entry pointed at an AML environment
that no longer exists (`az ml environment show` came back empty) -- `ensure_environment`
treats that as stale and rebuilds rather than handing back a version that would fail at job
submission.

**`az` returned something that isn't a version.** `az ml` tries to auto-upgrade itself and
cannot on an AML compute instance — the `ml` extension there is installed system-wide at
`/opt/az/extensions/ml`, owned by root, while you run as `azureuser`. It fails with
`Permission denied` and leaves the extension **half-deleted**, after which every `az ml`
command breaks. Run these steps from your laptop, or reinstall the extension with `sudo`.

**Where is the build status?** Only in Studio — the link each `ensure_environment` call
prints when it just built. An AML v2 environment build is an **ACR task run, not an AML
job**, so nothing in `az ml job list` shows it, and the v2 `Environment` object carries no
build state at all. `az ml environment show` proves the *asset* is registered, never that the
image is built.

**`verify_image` says `image_ref_has_spec44: False`.** The registered image was built from a
pre-spec-44 fsd. Re-run the relevant Part from a current checkout.

**A build failed.** Studio → Environments → the version → build log. Re-running the cell
tries the same digest again, so fix the underlying problem (an unreachable base image, a
broken `pip install`) before re-running, rather than expecting a fresh attempt for free.

**This notebook is committed — keep it clean.** Before `git add`, run *Kernel → Restart &
Clear All Outputs*. Its saved outputs would otherwise carry your subscription id, tenant id,
workspace URL and local paths into a public repo. `tests/test_notebooks.py` fails the build
if outputs or hardcoded identifiers get in, so a leak cannot land silently — but clearing them
yourself is faster than finding out from a red test.